In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import time
import datetime
import requests
import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By

### Selenium

In [66]:
# 크롬 드라이버 경로 설정
path = r"C:\Users\NT551_11TH\Downloads\chromedriver-win64\chromedriver.exe"

In [ ]:
# 카드 1개 상세 페이지
url = 'https://www.card-gorilla.com/card/detail/2885'


In [57]:
s = Service(path)
driver = webdriver.Chrome(service=s)
driver.get(url)

In [ ]:
# //*[@id="q-app"]/section/div[1]/section/div/article[2]/div[1]/dl[1]
# //*[@id="q-app"]/section/div[1]/section/div/article[2]/div[1]/dl[2]

In [68]:
benefit_list = []

In [73]:
# 할인 정보 데이터 개수 몇 개인지
benefit_path = '//*[@id="q-app"]/section/div[1]/section/div/article[2]/div[1]'
benefit_cnt = int(driver.find_element(By.XPATH, benefit_path).get_attribute('childElementCount'))

for cnt in range(1, benefit_cnt+1):
    benefit_btn = f'//*[@id="q-app"]/section/div[1]/section/div/article[2]/div[1]/dl[{cnt}]/dt'
    element = driver.find_element(By.XPATH, benefit_btn)
    # driver.find_element(By.XPATH, benefit_btn).click()
    driver.execute_script("arguments[0].click();", element)
    time.sleep(0.5)

    benefit_info_path = f'//*[@id="q-app"]/section/div[1]/section/div/article[2]/div[1]/dl[{cnt}]/dd'
    print(driver.find_element(By.XPATH, benefit_info_path).text)
    # benefit_list.append(driver.find_element(By.XPATH, benefit_info_path).text)

선택 옵션에 따른 할인 혜택 제공 (택 1)
- 국내 가맹점 0.7% 할인
- 아파트 관리비/통신 10% 할인
- 교육 10% 할인

  * 선택한 옵션에 대해서만 제공
  * 옵션은 모니모 앱, 삼성카드 홈페이지(PC, 모바일)/앱을 통해 매월 변경 가능하며, 변경 신청 다음 달 1일에 자동 반영
국내 가맹점 0.7% 할인
- 전월 이용금액에 관계없이, 할인한도 없이 국내 가맹점 0.7% 결제일할인
아파트 관리비/통신 10% 할인
- 아파트 관리비/통신요금 정기결제 시 10% 결제일할인
구분 할인 대상
아파트 관리비 아파트 관리비
이동통신 SKT/KT/LG U+/알뜰폰 (SK텔링크, KT스카이라이프, KT M 모바일, 헬로모바일, 미디어로그) 이동통신요금
인터넷, 유선통신 SK브로드밴드 /KT/KT스카이라이프/LG U+ 인터넷/유선통신요금

전월 이용금액대별 통합 월 할인한도
40만원 이상 80만원 이상 120만원 이상
7,000원 10,000원 15,000원
- 발급월 +1개월까지는 전월 이용금액 40만원 미만 시에도 40만원 이상 ~ 80만원 미만 실적구간 혜택 제공(전월 이용금액 80만원 이상 시에는 해당 실적구간 혜택 제공)
- 알뜰폰은 결합상품요금, 휴대폰 등 단말기 구매금액 및 대리점 카드 결제건 제외
- 인터넷, 유선통신은 사물인터넷(IoT)관련 요금, 휴대폰 등 단말기 구매금액 및 대리점 카드 결제건 제외
교육 10% 할인
- 교육 10% 결제일할인
구분 할인 대상
학원 입시/보습외국어/예체능계 학원
인터넷강의 이투스, 메가스터디교육(메가스터디, 엠베스트 엘리하이), 대성마이맥, 천재교과서(밀크T)
학습지 웅진씽크빅, 교원, 대교, 한솔교육

전월 이용금액대별 통합 월 할인한도
40만원 이상 80만원 이상 120만원 이상
7,000원 10,000원 15,000원
- 발급월 +1개월까지는 전월 이용금액 40만원 미만 시에도 40만원 이상 ~ 80만원 미만 실적구간 혜택 제공(전월 이용금액 80만원 이상 시에는 해당 실적구간 혜택 제공)
- 학원은

In [ ]:
# 테스트용
btn = '//*[@id="q-app"]/section/div[1]/section/div/article[2]/div[1]/dl[1]/dt'
driver.find_element( By.XPATH, btn).click()

test = '//*[@id="q-app"]/section/div[1]/section/div/article[2]/div[1]/dl[1]/dd'
driver.find_element(By.XPATH, test).text
# driver.find_element(By.XPATH, test).text.split('\n')

'선택 옵션에 따른 할인 혜택 제공 (택 1)\n- 국내 가맹점 0.7% 할인\n- 아파트 관리비/통신 10% 할인\n- 교육 10% 할인\n\n  * 선택한 옵션에 대해서만 제공\n  * 옵션은 모니모 앱, 삼성카드 홈페이지(PC, 모바일)/앱을 통해 매월 변경 가능하며, 변경 신청 다음 달 1일에 자동 반영'

In [ ]:
# //*[@id="q-app"]/section/div[1]/section/div/article[2]/div[1]/dl[2]/dd

### API 호출

In [43]:
card_data_list = []


In [ ]:
# raw_data.get('corp')['name']

'신한카드'

In [ ]:
# url = f"https://api.card-gorilla.com:8080/v1/cards/{1}"
# headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'}
    
# response = requests.get(url, headers=headers)
# # response

# raw_data = response.json()
# raw_data
# # card_info = {
# #     '카드번호': card_id,
# #     '카드명': raw_data.get('name', '이름없음'),       # name 키가 없으면 '이름없음' 반환
# #     '카드사': raw_data.get('corp_name', '알수없음'), # 예: 신한카드
# #     '카드타입': raw_data.get('type', '알수없음')     # 예: 신용/체크
# # }
            
# # card_data_list.append(card_info)

{'idx': 1,
 'cid': '001',
 'cate': 'CRD',
 'corp': {'idx': 2,
  'name': '신한카드',
  'name_eng': 'shinhan',
  'color': '#2252FF',
  'logo_img': {'name': 'logo_sh.png',
   'url': 'https://d1c5n4ri2guedi.cloudfront.net/corp/2/logo_img/33324/logo_sh.png'},
  'pr_container': None,
  'pr_detail_img': [],
  'pr_detail': '<!-- 소스코드 시작 --><div class="modal-promo"><div class="btn-close"><br></div></div><!-- 소스코드 끝 -->',
  'pr_container_chk': None,
  'pr_detail_img_chk': [],
  'pr_detail_chk': None,
  'tips': [{'title': '고릴라 TIP', 'contents': ''}],
  'is_event': True,
  'is_visible': True},
 'name': '신한카드 Hi-Point',
 'brand': [{'idx': 1,
   'name': 'VISA',
   'code': 'VISA',
   'logo_img': {'name': 'brand_visa.png',
    'url': 'https://d1c5n4ri2guedi.cloudfront.net/brand/1/logo_img/44542/brand_visa.png'},
   'is_visible': True},
  {'idx': 4,
   'name': 'Mastercard',
   'code': 'Mastercard',
   'logo_img': {'name': 'brand_mc.png',
    'url': 'https://d1c5n4ri2guedi.cloudfront.net/brand/4/logo_img/44

In [44]:
for card_id in tqdm.tqdm(range(1, 3000)):
    url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'}
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            raw_data = response.json()

            card_info = {
                '카드번호': card_id,
                '카드명': raw_data.get('name', '이름없음'),       # name 키가 없으면 '이름없음' 반환
                '카드사': raw_data.get('corp').get('name', '알수없음'),  # 예: 신한카드
                '카드타입': raw_data.get('cate', '알수없음')     # 예: 신용/체크
                }
            
            card_data_list.append(card_info)
            print(f"[성공] {card_id}번 카드 ({card_info['카드명']}) 수집 완료")

        elif response.status_code == 404:
            print(f"[Skip] {card_id}번 카드 정보가 존재하지 않습니다.")
        else:
            print(f"[Error] {card_id}번 카드 정보 요청 실패 (상태 코드: {response.status_code})")
    
    except Exception as e:
        print(f"[오류] {card_id}번 카드 정보 요청 중 예외 발생: {e}")

    time.sleep(1)

  0%|          | 0/2999 [00:00<?, ?it/s]

[성공] 1번 카드 (신한카드 Hi-Point) 수집 완료


  0%|          | 1/2999 [00:01<57:31,  1.15s/it]

[성공] 2번 카드 (신한카드 Love) 수집 완료


  0%|          | 2/2999 [00:02<56:43,  1.14s/it]

...

100%|█████████▉| 2997/2999 [58:00<00:02,  1.08s/it]

[Skip] 2998번 카드 정보가 존재하지 않습니다.


100%|█████████▉| 2998/2999 [58:01<00:01,  1.09s/it]

[Skip] 2999번 카드 정보가 존재하지 않습니다.


100%|██████████| 2999/2999 [58:02<00:00,  1.16s/it]


In [45]:
df = pd.DataFrame(card_data_list)

In [ ]:
# df.to_csv('card_data.csv', index=False, encoding='utf-8-sig')

### 데이터 불러오기

In [12]:
df = pd.read_csv('card_data.csv', index_col=0)
df.head()

,카드명,카드사,카드타입
카드번호,,,
1,신한카드 Hi-Point,신한카드,CRD
2,신한카드 Love,신한카드,CRD
3,신한카드 Lady,신한카드,CRD
4,SK에너지 신한카드The you,신한카드,CRD
8,신한카드 The CLASSIC-Y,신한카드,CRD


In [13]:
df.tail()

,카드명,카드사,카드타입
카드번호,,,
2974,현대카드 하이브리드(Apple Pay Rewards),현대카드,CHK
2975,LOCA LIKIT 1.5,롯데카드,CRD
2976,우체국 MY-TYPE 체크카드,우체국,CHK
2977,신한카드 SOL트립앤J 체크,신한카드,CHK
2978,카드의정석2 SHOPPER,우리카드,CRD


In [14]:
len(df)

2810

In [15]:
df.shape

(2810, 3)

In [16]:
df.info()

<class 'pandas.DataFrame'>
Index: 2810 entries, 1 to 2978
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   카드명     2810 non-null   str  
 1   카드사     2810 non-null   str  
 2   카드타입    2810 non-null   str  
dtypes: str(3)
memory usage: 87.8 KB


In [17]:
df.isnull().sum()

카드명     0
카드사     0
카드타입    0
dtype: int64

In [21]:
df['카드사'].unique()

<StringArray>
[        '신한카드',         '삼성카드',         '우리카드',         '씨티카드',
       'KB국민카드',       'NH농협카드',         '롯데카드',         '하나카드',
      'IBK기업은행',       'SC제일은행',         '현대카드', 'SSGPAY. CARD',
        '카카오뱅크',         '케이뱅크',          '우체국',         'iM뱅크',
      'BNK부산은행',         '광주은행',       'Sh수협은행',           '신협',
         '교보증권',        '유안타증권',      'KDB산업은행',      'SBI저축은행',
      'MG새마을금고',         '제주은행',         '전북은행',        '카카오페이',
      'BC 바로카드',       '유진투자증권',         'SK증권',       '미래에셋증권',
       'NH투자증권',         'KB증권',       'DB금융투자',       '한국투자증권',
         '토스뱅크',     '엔에이치엔페이코',         '코나카드',        '아이오로라',
           '토스',          '한패스',           '차이',         '핀크카드',
           '다날',        '트래블월렛',      'KG 파이낸셜',           '핀트',
        '현대백화점',      'BNK경남은행',        '네이버페이',         '머니트리',
        'OK캐쉬백',       '웰컴저축은행']
Length: 54, dtype: str

In [18]:
df['카드사'].value_counts()

카드사
롯데카드            548
삼성카드            371
KB국민카드          285
우리카드            277
신한카드            251
현대카드            217
NH농협카드          201
하나카드            184
IBK기업은행         116
BNK부산은행          44
BC 바로카드          43
iM뱅크             41
MG새마을금고          31
우체국              26
광주은행             24
씨티카드             19
신협               14
제주은행             14
전북은행             14
SC제일은행           10
케이뱅크             10
Sh수협은행            7
카카오뱅크             5
SBI저축은행           4
KB증권              4
코나카드              4
현대백화점             4
SSGPAY. CARD      3
교보증권              3
유안타증권             3
카카오페이             2
유진투자증권            2
SK증권              2
미래에셋증권            2
DB금융투자            2
토스뱅크              2
한패스               2
차이                2
OK캐쉬백             2
KDB산업은행           1
NH투자증권            1
한국투자증권            1
엔에이치엔페이코          1
아이오로라             1
토스                1
핀크카드              1
다날                1
트래블월렛             1
KG 파이낸셜           1
핀트              

In [22]:
card_comp_list = ['삼성카드', '신한카드', '현대카드', 'KB국민카드', '롯데카드', 
                  '우리카드', '하나카드', 'NH농협카드', 'IBK기업은행', 'BC 바로카드']

In [25]:
top_df = df.loc[df['카드사'].isin(card_comp_list)]
len(top_df)

2493

In [30]:
valid_ids = top_df.index.tolist()
len(valid_ids)

2493

### 정보 추가

In [69]:
card_data_list = []

In [70]:
for card_id in tqdm.tqdm(valid_ids):
    url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'}
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            raw_data = response.json()

            # 혜택 카테고리 추출 (key_benefit 활용 + 중복 제거)
            category_list = []
            if 'key_benefit' in raw_data and raw_data['key_benefit']:
                for benefit in raw_data['key_benefit']:
                    if 'cate' in benefit and 'name' in benefit['cate']:
                        cate_name = benefit['cate']['name']

                        if cate_name not in ['기타', '유의사항']:
                            if cate_name not in category_list:
                                category_list.append(cate_name)

            # 주요 혜택 요약 추출 (top_benefit 활용)
            top_tags_list = []
            if 'top_benefit' in raw_data and raw_data['top_benefit']:
                for top in raw_data['top_benefit']:
                    if 'tags' in top and top['tags']:
                        top_tags_list.append(" ".join(top['tags']))

            # 현재 발급 가능 여부
            is_discontinued = raw_data.get('is_discon', False)
            issuable_status = 'X' if is_discontinued else 'O'

            card_info = {
                '카드번호': card_id,
                '카드명': raw_data.get('name', '이름없음'),       # name 키가 없으면 '이름없음' 반환
                '카드사': raw_data.get('corp').get('name', '알수없음'),  # 예: 신한카드
                '카드타입': raw_data.get('cate', '알수없음'),     # 예: 신용/체크
                '출시일': raw_data.get('release_dt', '알수없음'),
                '전월실적': raw_data.get('pre_month_money', 0),
                '카테고리': ", ".join(category_list),
                '주요혜택': ", ".join(top_tags_list),
                '발급가능여부': issuable_status
                }
            
            card_data_list.append(card_info)
            # print(f"[성공] {card_id}번 카드 ({card_info['카드명']}) 수집 완료")

        elif response.status_code == 404:
            # print(f"[Skip] {card_id}번 카드 정보가 존재하지 않습니다.")
            pass
        else:
            print(f"[Error] {card_id}번 카드 정보 요청 실패 (상태 코드: {response.status_code})")
    
    except Exception as e:
        print(f"[오류] {card_id}번 카드 정보 요청 중 예외 발생: {e}")

    # time.sleep(1)

100%|██████████| 2493/2493 [06:46<00:00,  6.14it/s]


In [71]:
df = pd.DataFrame(card_data_list)
df.head()

,카드번호,카드명,카드사,카드타입,출시일,전월실적,카테고리,주요혜택,발급가능여부
0,1,신한카드 Hi-Point,신한카드,CRD,,0,"쇼핑, 모든가맹점, 주유, 금융, 통신, 적립","전국가맹점 0.2~2.0% 적립, 특정가맹점 1~5% 적립, 에이치디현대오일뱅크 6...",X
1,2,신한카드 Love,신한카드,CRD,,200000,"백화점, 패밀리레스토랑, 주유소, 영화, 적립, 테마파크, 경기관람","백화점/할인점 5% 할인, 스타벅스 20% 할인, GS칼텍스 60원/L 할인",X
2,3,신한카드 Lady,신한카드,CRD,,300000,"쇼핑, 푸드, 영화, 주유소, 교육/육아, 테마파크, 통신, 경기관람","3대백화점 5% 할인, 패밀리레스토랑 20% 할인, 놀이공원 30~50% 할인",X
3,4,SK에너지 신한카드The you,신한카드,CRD,,300000,"주유소, 대중교통, 무이자할부, 테마파크, 영화, 헤어, 경기관람, 여행사","SK주유소 100원/L 할인, 지하철,버스,택시 3~7% 할인, 놀이공원 30~50...",X
4,8,신한카드 The CLASSIC-Y,신한카드,CRD,,0,"선택형, 모든가맹점, 주유소, 생활, 면세점, 진에어, 공연/전시, 프리미엄","Gift Option 매년1회 선택이용, 마이신한포인트 0.7~5% 적립, GS칼텍...",O


In [ ]:
# df.loc[df['발급가능여부'].isnull()].head()

,카드번호,카드명,카드사,카드타입,출시일,전월실적,카테고리,주요혜택,발급가능여부
0,1,신한카드 Hi-Point,신한카드,CRD,,0,"쇼핑, 모든가맹점, 주유, 금융, 통신, 적립","전국가맹점 0.2~2.0% 적립, 특정가맹점 1~5% 적립, 에이치디현대오일뱅크 6...",NaN
1,2,신한카드 Love,신한카드,CRD,,200000,"백화점, 패밀리레스토랑, 주유소, 영화, 적립, 테마파크, 경기관람","백화점/할인점 5% 할인, 스타벅스 20% 할인, GS칼텍스 60원/L 할인",NaN
2,3,신한카드 Lady,신한카드,CRD,,300000,"쇼핑, 푸드, 영화, 주유소, 교육/육아, 테마파크, 통신, 경기관람","3대백화점 5% 할인, 패밀리레스토랑 20% 할인, 놀이공원 30~50% 할인",NaN
3,4,SK에너지 신한카드The you,신한카드,CRD,,300000,"주유소, 대중교통, 무이자할부, 테마파크, 영화, 헤어, 경기관람, 여행사","SK주유소 100원/L 할인, 지하철,버스,택시 3~7% 할인, 놀이공원 30~50...",NaN
4,8,신한카드 The CLASSIC-Y,신한카드,CRD,,0,"선택형, 모든가맹점, 주유소, 생활, 면세점, 진에어, 공연/전시, 프리미엄","Gift Option 매년1회 선택이용, 마이신한포인트 0.7~5% 적립, GS칼텍...",NaN
...,...,...,...,...,...,...,...,...,...
2488,2973,현대카드 하이브리드(캐시백형),현대카드,CHK,2026-04-28,0,"국내외가맹점, 적립, 하이브리드","국내외 가맹점 0.3% 캐시백, 일반음식점, 배달 앱 3% 캐시백, 편의점, 대중교...",NaN
2489,2974,현대카드 하이브리드(Apple Pay Rewards),현대카드,CHK,2026-04-28,300000,"간편결제, 하이브리드",Apple Pay 이용 금액 10% 캐시백,NaN
2490,2975,LOCA LIKIT 1.5,롯데카드,CRD,2026-04-07,0,"국내가맹점, 해외, 무이자할부","국내 가맹점 0.8% 할인, 해외 가맹점 1.5% 할인",NaN
2491,2977,신한카드 SOL트립앤J 체크,신한카드,CHK,2026-03-30,0,"해외이용, 해외, 할인","해외 이용 수수료 면제, 일본 돈키호테/편의점 50% 할인, 국내 편의점/대중교통 ...",NaN


In [ ]:
# df.loc[df['발급가능여부'] == 'X'].head()

,카드번호,카드명,카드사,카드타입,출시일,전월실적,카테고리,주요혜택,발급가능여부
2493,1,신한카드 Hi-Point,신한카드,CRD,,0,"쇼핑, 모든가맹점, 주유, 금융, 통신, 적립","전국가맹점 0.2~2.0% 적립, 특정가맹점 1~5% 적립, 에이치디현대오일뱅크 6...",X
2494,2,신한카드 Love,신한카드,CRD,,200000,"백화점, 패밀리레스토랑, 주유소, 영화, 적립, 테마파크, 경기관람","백화점/할인점 5% 할인, 스타벅스 20% 할인, GS칼텍스 60원/L 할인",X
2495,3,신한카드 Lady,신한카드,CRD,,300000,"쇼핑, 푸드, 영화, 주유소, 교육/육아, 테마파크, 통신, 경기관람","3대백화점 5% 할인, 패밀리레스토랑 20% 할인, 놀이공원 30~50% 할인",X
2496,4,SK에너지 신한카드The you,신한카드,CRD,,300000,"주유소, 대중교통, 무이자할부, 테마파크, 영화, 헤어, 경기관람, 여행사","SK주유소 100원/L 할인, 지하철,버스,택시 3~7% 할인, 놀이공원 30~50...",X
2499,11,신한카드 Simple+,신한카드,CRD,,0,"모든가맹점, 생활, 영화","전가맹점 0.7% 캐시백, 이동통신요금 0.7% 추가캐시백, 생활친화가맹점 월 10...",X
...,...,...,...,...,...,...,...,...,...
4802,2762,THE iD. TITANIUM (아시아나),삼성카드,CRD,2024-08-26,0,"바우처, 아시아나항공, 프리미엄 서비스","바우처 연 2가지 제공, 1,500원당 2 마일리지 기본 적립, 여행/골프/면세점 ...",X
4837,2797,SKT T다운 하나카드,하나카드,CRD,,400000,SKT,SKT 라이트할부 청구할인,X
4838,2798,LG전자 스페셜 롯데카드,롯데카드,CRD,,300000,"할인, 캐시백, 멤버십포인트","렌탈료 최대 23,000원 결제일 할인, 장기할부이용 최대 23,000원 캐시백",X
4842,2802,LG U+ NU 우리카드 Ⅱ,우리카드,CRD,,0,LGU+,"LG U+ 통신요금 최대 20,000원 할인, LG U+ 단말기 장기할부 서비스",X


In [ ]:
# df.loc[df['발급가능여부'] == 'O'].head()

,카드번호,카드명,카드사,카드타입,출시일,전월실적,카테고리,주요혜택,발급가능여부
2497,8,신한카드 The CLASSIC-Y,신한카드,CRD,,0,"선택형, 모든가맹점, 주유소, 생활, 면세점, 진에어, 공연/전시, 프리미엄","Gift Option 매년1회 선택이용, 마이신한포인트 0.7~5% 적립, GS칼텍...",O
2498,10,신한카드 B.Big(삑),신한카드,CRD,,300000,"대중교통, 택시, 백화점, 카페, 편의점, 통신, 영화, 캐시백","대중교통 200~600원 할인, 택시 10% 할인, KTX 10% 할인",O
2500,12,신한카드 Air Platinum#,신한카드,CRD,,0,"항공마일리지, 생활, 프리미엄","1,500원당 1마일 적립, 1,000원당 1마일 적립",O
2501,13,신한카드 Mr.Life,신한카드,CRD,2015-09-03,300000,"공과금, 편의점, 병원/약국, 생활, 온라인쇼핑, 택시, 푸드, 대형마트, 주유소","공과금 10% 할인, 마트,편의점 10% 할인, 식음료 10% 할인",O
2502,14,신한카드 YOLO ⓘ,신한카드,CRD,,300000,"선택형, 카페, 편의점, 소셜커머스, 택시, 베이커리, 영화","6개가맹점 할인율 선택, 스타벅스 쿠폰 제공",O
...,...,...,...,...,...,...,...,...,...
4981,2973,현대카드 하이브리드(캐시백형),현대카드,CHK,2026-04-28,0,"국내외가맹점, 적립, 하이브리드","국내외 가맹점 0.3% 캐시백, 일반음식점, 배달 앱 3% 캐시백, 편의점, 대중교...",O
4982,2974,현대카드 하이브리드(Apple Pay Rewards),현대카드,CHK,2026-04-28,300000,"간편결제, 하이브리드",Apple Pay 이용 금액 10% 캐시백,O
4983,2975,LOCA LIKIT 1.5,롯데카드,CRD,2026-04-07,0,"국내가맹점, 해외, 무이자할부","국내 가맹점 0.8% 할인, 해외 가맹점 1.5% 할인",O
4984,2977,신한카드 SOL트립앤J 체크,신한카드,CHK,2026-03-30,0,"해외이용, 해외, 할인","해외 이용 수수료 면제, 일본 돈키호테/편의점 50% 할인, 국내 편의점/대중교통 ...",O


In [ ]:
df.to_csv('/data/card_data.csv', index=False, encoding='utf-8-sig')